# 06 — Normal versus shifted-lognormal vols, and the exact conversion

## Two ways to quote the same price

A swaption premium is an annuity times an undiscounted option price on the forward swap rate $F$. Two models turn a
price into a single number.

**Bachelier (normal).** $dF = \sigma_N\, dW$, and with $d = (F-K)/(\sigma_N\sqrt T)$

$$C_N = (F-K)\,\Phi(d) + \sigma_N\sqrt T\,\varphi(d), \qquad P_N = (K-F)\,\Phi(-d) + \sigma_N\sqrt T\,\varphi(d).$$

**Shifted Black.** $d(F+s) = \sigma_{LN}(F+s)\,dW$, and with $f = F+s$, $k = K+s$,
$d_{1,2} = [\ln(f/k) \pm \tfrac12\sigma_{LN}^2 T]/(\sigma_{LN}\sqrt T)$

$$C_{LN} = f\,\Phi(d_1) - k\,\Phi(d_2), \qquad P_{LN} = k\,\Phi(-d_2) - f\,\Phi(-d_1).$$

Both satisfy put–call parity $C - P = F - K$, so a price has one $\sigma_N$ and, if it is attainable, one
$\sigma_{LN}$ for a given shift, whichever option you started from.

**Why desks care.** A normal vol is roughly level-independent in bp and exists for any sign of $F$ and $K$; USD
screens quote it. A lognormal vol needs $F + s > 0$ and $K + s > 0$; EUR, CHF and JPY screens quote it with a
*published* shift (for example 1% or 3%), so that negative rates stay inside the model. The shift is a quoting
convention, never a parameter you tune.

## A hand example: the ATM series

At the money $d = 0$, so $C_N = \sigma_N\sqrt T/\sqrt{2\pi}$ and $C_{LN} = f\,(2\Phi(\sigma_{LN}\sqrt T/2) - 1)$.
Expanding $2\Phi(u/2) - 1 = \frac{u}{\sqrt{2\pi}}\left(1 - \frac{u^2}{24} + \frac{u^4}{640} - \dots\right)$ gives

$$\sigma_N \approx \sigma_{LN}\,(F+s)\left(1 - \frac{\sigma_{LN}^2 T}{24}\right), \qquad \text{relative error} \approx \frac{(\sigma_{LN}^2T)^2}{640}.$$

With $F = 3.5\%$, $s = 0$, $\sigma_{LN} = 25\%$ and $T = 1$ you get $\sigma_N \approx 0.25 \times 0.035 \times (1 - 0.0625/24) = 87.27$ bp.
This cell needs no C++.

In [ ]:
import _util
from _util import run_demo
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import norm

F, s_ln, T = 0.035, 0.25, 1.0
atm_black = F * (2 * norm.cdf(s_ln * np.sqrt(T) / 2) - 1)   # ATM shifted-Black price, s = 0
s_n_exact = atm_black * np.sqrt(2 * np.pi) / np.sqrt(T)      # the Bachelier ATM price inverts in closed form
s_n_series = s_ln * F * (1 - s_ln**2 * T / 24)
print(f"ATM Black price   {atm_black:.8f}")
print(f"sigma_N exact     {s_n_exact * 1e4:.4f} bp")
print(f"sigma_N series    {s_n_series * 1e4:.4f} bp   relative error {abs(s_n_series / s_n_exact - 1):.2e}, (s^2 T)^2/640 = {(s_ln**2 * T)**2 / 640:.2e}")

## One smile, two quotes

`rl_demo_vol smile` evaluates Hagan's SABR expansion in its normal form (B.69) and its lognormal form (2.17a). Both
are closed forms. A smile that looks gently convex in normal vol is a steep negative skew in lognormal vol:
most of the lognormal "skew" in rates is the $1/(F+s)$ scaling, not information about the distribution.

In [ ]:
sm = run_demo("rl_demo_vol", "smile", 0.0459, 0.5, -0.13, 0.41, 0.0, 0.036, 1.0, 0.016, 0.056, 81)
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(sm["strike"] * 100, sm["normal_vol_bp"], label="normal vol (bp, left)")
ax1.set_xlabel("strike, %"); ax1.set_ylabel("normal vol, bp")
ax2 = ax1.twinx()
ax2.plot(sm["strike"] * 100, sm["lognormal_vol_pct"], "C1--", label="lognormal vol (%, right)")
ax2.set_ylabel("lognormal vol, %")
fig.legend(loc="upper center"); ax1.set_title("1Yx10Y SABR smile (beta 0.5), quoted both ways"); plt.show()

## The exact conversion

`vol::normalToShiftedLognormalVol` and `vol::shiftedLognormalToNormalVol` live in
(`cpp/vol/src/conversion.cpp`). The recipe is short and every line of it is a decision:

1. Price the **out-of-the-money** option under the source model. The in-the-money price is intrinsic plus time
   value, and subtracting a 300 bp intrinsic destroys the digits the inversion needs.
2. Check the domain: $F + s > 0$, $K + s > 0$, and attainability. The OTM lognormal price is capped by
   $\min(F+s, K+s)$; a normal price above that cap has no lognormal vol.
3. Invert under the target model with the implied-vol solvers.

In the cells below, the
first plot shows what a flat 85 bp normal smile looks like to a lognormal trader at three shifts, and the round trip
must be below 1e-12 in vol.

In [ ]:
F, T, sn = 0.036, 2.0, 0.0085
strikes = np.linspace(0.011, 0.066, 23)
fig, ax = plt.subplots(figsize=(9, 4))
worst = 0.0
for shift in (0.0, 0.01, 0.03):
    rows = pd.concat([run_demo("rl_demo_vol", "convert", F, k, T, sn, shift) for k in strikes if k + shift > 1e-4])
    ax.plot(rows["strike"] * 100, rows["lognormal_vol_pct"], marker="o", ms=3, label=f"shift {shift:.0%}")
    worst = max(worst, float((rows["roundtrip_normal_vol_bp"] - rows["normal_vol_bp"]).abs().max()))
ax.set_xlabel("strike, %"); ax.set_ylabel("shifted lognormal vol, %"); ax.legend()
ax.set_title("A flat 85 bp normal smile, quoted in shifted lognormal"); plt.show()
print(f"max round-trip error: {worst:.2e} bp (target < 1e-8 bp)")

## How good is the ATM series?

The series error grows like $(\sigma_{LN}^2 T)^2/640$. At 2Y it is negligible; at 30Y on a low-rate currency it is
a visible fraction of a bp, which is why a production conversion is exact and the series is only an initial guess.

In [ ]:
rows = []
for T in (0.25, 1, 2, 5, 10, 20, 30):
    r = run_demo("rl_demo_vol", "convert", 0.036, 0.036, T, 0.0085, 0.0).iloc[0]
    exact, series = r["lognormal_vol_pct"] / 100, r["atm_series_lognormal_pct"] / 100
    rows.append((T, exact, series, abs(series / exact - 1), (exact**2 * T) ** 2 / 640))
atm = pd.DataFrame(rows, columns=["T", "exact", "series", "rel_error", "u4_over_640"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(atm["T"], atm["rel_error"], "o-", label="series vs exact")
ax.loglog(atm["T"], atm["u4_over_640"], "--", label="(sigma^2 T)^2 / 640")
ax.set_xlabel("expiry, years"); ax.set_ylabel("relative error"); ax.legend(); plt.show()
atm

## What to take away

* A vol is a price in a unit. Conversions are exact or they are a calibration residual nobody can explain.
* Invert the OTM option. Check the shifted domain and attainability before the solver, and say in the error which
  one failed.
* The shift is a market convention attached to the quote. Changing it changes the lognormal number, never the price.